In [ ]:
!pip install --upgrade pip
!pip install -q kokoro>=0.9.2 soundfile speechbrain sarvamai
!apt-get -qq -y install espeak-ng > /dev/null 2>&1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.1.1 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
libcugraph-cu12 25.6.0 requires libraft-cu12==25.6.*, but you have libraft-cu12 25.2.0 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.0a1 which is incompatible.
pylibcugraph-cu12 25.6.0 requires pylibraft-cu12==25.6.*, but you have pylibraft-cu12 25.2.0 which is incompatible.
pylibcugraph-cu12 25.6.0 requires rmm-cu12==25.6.*, but you have rmm-cu12 25.2.0 which is incompatible.


In [ ]:
!git clone https://github.com/sathishkumar67/Kokoro-Indian-Voice.git
!mv Kokoro-Indian-Voice/* ./
!rm -rf Kokoro-Indian-Voice

In [ ]:
# Standard library imports
import os
import glob
import random

# PyTorch core and convenience modules
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Audio I/O and processing
import torchaudio
import soundfile as sf

# Pretrained models / project pipelines
from speechbrain.pretrained import EncoderClassifier
from kokoro import KPipeline

# Notebook display helpers
from IPython.display import display, Audio

# Device selection: prefer CUDA if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
encoder = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", run_opts={"device": device})

def get_speaker_emb(path):
    wav, sr = torchaudio.load(path)
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)

    wav = wav.to(device)
    with torch.no_grad():
        emb = encoder.encode_batch(wav)  # internally handles lengths
    emb = emb.squeeze(0).detach().cpu()  # [256]
    return emb # [1, 192]